# Track A — Full Evaluation Notebook
## 5G RF Drive-Test Troubleshooting · Accuracy & F1 Assessment

This notebook runs the **Track A pipeline end-to-end** with full observability:

- Builds the **RAG index** from `train.json` (80 % split) so few-shot examples are always active
- Runs the LangGraph pipeline on an **eval split** (20 %) with ground-truth labels available
- Displays the **full LLM thinking trace** for each scenario
- Computes **Exact-Match Accuracy**, **Macro F1**, and **Micro F1** (multi-label aware)
- Breaks down metrics **per tag** (`single-answer` / `multiple-answer`) and **per problem type**
- Runs inference on the **full test set** and exports `result_track_a.csv`

---

### Notebook structure

| Section | Description |
|---------|-------------|
| §1 Platform & Repo Setup | Detect Colab / Kaggle / local, clone repo |
| §2 Install Dependencies | `pip install -r requirements.txt` |
| §3 Configuration | `TrackAConfig` dataclass — edit once, applies everywhere |
| §4 Apply Configuration | Push config to `os.environ` + patch `src.config.settings` |
| §5 Ollama Setup | Install daemon + pull model (dev mode) |
| §6 Dataset & Tool Server | Download dataset, start FastAPI server |
| §7 Train / Eval Split | 80/20 stratified split of `train.json` for RAG vs evaluation |
| §8 Build RAG Index | KNN index on the 80 % training slice |
| §9 Compile LangGraph | Assemble the Track A pipeline |
| §10 Eval Run — With Thinking Traces | Run pipeline on 20 % eval set, display reasoning |
| §11 Evaluation Metrics | Accuracy, Macro F1, Micro F1 — overall and per-tag |
| §12 Per-Problem-Type Analysis | Breakdown by LATE_HANDOVER, NEIGHBOR_MISSING, UNKNOWN, … |
| §13 Error Analysis | Inspect wrong predictions in detail |
| §14 Full Test-Set Inference | Run on `test.json` (no labels) |
| §15 Export Results | Write `result_track_a.csv` |

## §1 — Platform Setup & Repository Clone

In [ ]:
"""
§1 — Platform detection, repo clone, sys.path setup.
Works transparently on Colab, Kaggle, and local Jupyter.
"""
import os
import sys
from pathlib import Path

# ── Detect platform ──────────────────────────────────────────────────────────
try:
    import google.colab  # noqa: F401
    PLATFORM = "colab"
except ImportError:
    PLATFORM = "kaggle" if os.path.exists("/kaggle/working") else "local"

print(f"Platform : {PLATFORM}")

# ── GitHub PAT ────────────────────────────────────────────────────────────────
GITHUB_PAT: str = ""
if PLATFORM == "colab":
    try:
        from google.colab import userdata
        GITHUB_PAT = userdata.get("GITHUB_PAT") or ""
    except Exception:
        pass
elif PLATFORM == "kaggle":
    try:
        from kaggle_secrets import UserSecretsClient
        GITHUB_PAT = UserSecretsClient().get_secret("GITHUB_PAT") or ""
    except Exception:
        pass
else:
    GITHUB_PAT = os.getenv("GITHUB_PAT", "")

# ── Paths ─────────────────────────────────────────────────────────────────────
_REPO_SLUG = "Seydifa/Troubleshooting-Agentic"

if PLATFORM == "colab":
    REPO_DIR = Path("/content/Troubleshooting-Agentic")
elif PLATFORM == "kaggle":
    REPO_DIR = Path("/kaggle/working/Troubleshooting-Agentic")
else:
    # Running locally inside the repo — parent of notebooks/
    REPO_DIR = (
        Path(__file__).resolve().parent.parent
        if "__file__" in dir()
        else Path.cwd().parent
    )

REPO_DIR = REPO_DIR.resolve()

# ── Clone / pull (cloud only) ─────────────────────────────────────────────────
if PLATFORM in ("colab", "kaggle"):
    _auth = f"https://{GITHUB_PAT}@github.com/{_REPO_SLUG}.git" if GITHUB_PAT else f"https://github.com/{_REPO_SLUG}.git"
    if not REPO_DIR.exists():
        print(f"Cloning {_REPO_SLUG} → {REPO_DIR} …")
        ret = os.system(f"git clone {_auth} {REPO_DIR}")
        if ret != 0:
            raise RuntimeError("git clone failed — check your GITHUB_PAT.")
    else:
        os.system(f"git -C {REPO_DIR} remote set-url origin {_auth}")
        os.system(f"git -C {REPO_DIR} pull --ff-only")
        print(f"Repo pulled: {REPO_DIR}")
else:
    print(f"Local run — repo at {REPO_DIR}")

# ── Activate repo root in sys.path + set cwd ─────────────────────────────────
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

os.chdir(REPO_DIR)
print(f"Working directory : {Path.cwd()}")
print("Done.")

## §2 — Install Dependencies

In [ ]:
import subprocess

def pip_install(*packages: str) -> None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        stdout=subprocess.DEVNULL,
    )

print("Installing project dependencies …")
pip_install("-r", "requirements.txt")
pip_install("fastapi", "uvicorn[standard]", "scikit-learn", "pandas", "matplotlib", "seaborn")
print("All dependencies installed.")

## §3 — Configuration

> **Edit only this cell.**  
> Set `ENV = "dev"` for Ollama (free, local) or `"prod"` for OpenRouter (cloud).  
> `EVAL_SPLIT` controls the fraction of `train.json` reserved for evaluation (the rest feeds the RAG index).

In [ ]:
from dataclasses import dataclass
from typing import Literal, Optional


@dataclass
class TrackAConfig:
    """All runtime parameters for the Track A evaluation notebook.

    Edit the fields between the two dashed lines, then run §4 to apply.
    """

    # ── LLM backend ───────────────────────────────────────────────────────────
    # "dev"  → Ollama  (free, local / Colab)
    # "prod" → OpenRouter (requires openrouter_api_key)
    ENV: Literal["dev", "prod"] = "dev"

    # ── Model ─────────────────────────────────────────────────────────────────
    MODEL_NAME: str            = "qwen3.5:35b-a3b"   # dev model (Ollama)
    PROD_MODEL_NAME: str       = "Qwen/Qwen3.5-35B-A3B"  # prod model

    # ── Endpoints ─────────────────────────────────────────────────────────────
    OLLAMA_BASE_URL: str       = "http://localhost:11434"
    OPENROUTER_API_KEY: str    = ""
    OPENROUTER_BASE_URL: str   = "https://openrouter.ai/api/v1"
    TOOL_SERVER_URL: str       = "http://localhost:8000"

    # ── Paths ─────────────────────────────────────────────────────────────────
    TRAIN_FILE: str            = "data/raw/Track A/data/Phase_1/train.json"
    TEST_FILE: str             = "data/raw/Track A/data/Phase_1/test.json"
    RAG_INDEX: str             = "rag_index_track_a.pkl"
    TOOL_CACHE: str            = "tool_cache_track_a.pkl"
    OUTPUT_CSV: str            = "result_track_a.csv"

    # ── HuggingFace (dataset download) ────────────────────────────────────────
    HF_TOKEN: str              = ""

    # ── Evaluation parameters ─────────────────────────────────────────────────
    # Fraction of train.json reserved for evaluation (rest → RAG index)
    EVAL_SPLIT: float          = 0.20
    # Random seed for reproducible split
    SPLIT_SEED: int            = 42
    # Max scenarios to index in RAG (set None to use all train examples)
    RAG_MAX_EXAMPLES: Optional[int] = None
    # KNN neighbours retrieved per query
    RAG_K: int                 = 3
    # True → force rebuild even if RAG_INDEX already exists on disk
    FORCE_REBUILD_RAG: bool    = False

    # ── Run limits (None = full set) ──────────────────────────────────────────
    # Limit evaluation set for quick smoke-test (None = full eval set)
    EVAL_LIMIT: Optional[int]  = None
    # Limit test set inference (None = all)
    TEST_LIMIT: Optional[int]  = None

    # ── Verbosity ─────────────────────────────────────────────────────────────
    # True → print full LLM thinking trace per scenario
    SHOW_THINKING: bool        = True
    # True → emit DEBUG-level logs (very verbose)
    DEBUG_LOGS: bool           = False

    def summary(self) -> None:
        print("=" * 60)
        print("  TrackAConfig")
        print("=" * 60)
        print(f"  ENV              : {self.ENV}")
        if self.ENV == "dev":
            print(f"  Model            : {self.MODEL_NAME}  (Ollama @ {self.OLLAMA_BASE_URL})")
        else:
            key_preview = (self.OPENROUTER_API_KEY[:6] + "…") if self.OPENROUTER_API_KEY else "NOT SET"
            print(f"  Model            : {self.PROD_MODEL_NAME}  (OpenRouter, key={key_preview})")
        print(f"  Tool server      : {self.TOOL_SERVER_URL}")
        print(f"  Eval split       : {self.EVAL_SPLIT:.0%}  (seed={self.SPLIT_SEED})")
        print(f"  RAG k            : {self.RAG_K}")
        print(f"  Eval limit       : {self.EVAL_LIMIT or 'ALL'}")
        print(f"  Test limit       : {self.TEST_LIMIT or 'ALL'}")
        print(f"  Show thinking    : {self.SHOW_THINKING}")
        print(f"  Output CSV       : {self.OUTPUT_CSV}")
        print("=" * 60)


# ── EDIT BELOW THIS LINE ─────────────────────────────────────────────────────

cfg = TrackAConfig()

cfg.ENV             = "dev"
cfg.MODEL_NAME      = "qwen3.5:35b-a3b"
cfg.EVAL_SPLIT      = 0.20          # 20 % of train.json used for evaluation
cfg.EVAL_LIMIT      = None          # None → full eval set
cfg.TEST_LIMIT      = None          # None → full test set
cfg.SHOW_THINKING   = True          # display reasoning traces in §10
cfg.DEBUG_LOGS      = False
cfg.HF_TOKEN        = ""            # fill in if dataset not yet downloaded
cfg.OPENROUTER_API_KEY = ""         # fill in for prod mode

# ── END EDIT ZONE ─────────────────────────────────────────────────────────────

cfg.summary()

## §4 — Apply Configuration

Pushes every `TrackAConfig` field to `os.environ` and patches `src.config.settings` so every downstream module picks up the correct values without a kernel restart.

In [ ]:
import importlib
import logging

# ── 1. Write to os.environ ────────────────────────────────────────────────────
_active_model = cfg.PROD_MODEL_NAME if cfg.ENV == "prod" else cfg.MODEL_NAME

_env_map = {
    "ENV":                 cfg.ENV,
    "MODEL_NAME":          _active_model,
    "PARSER_MODEL_NAME":   _active_model,
    "OLLAMA_BASE_URL":     cfg.OLLAMA_BASE_URL,
    "OPENROUTER_API_KEY":  cfg.OPENROUTER_API_KEY,
    "OPENROUTER_BASE_URL": cfg.OPENROUTER_BASE_URL,
    "TOOL_SERVER_URL":     cfg.TOOL_SERVER_URL,
    "HF_TOKEN":            cfg.HF_TOKEN,
}

for key, value in _env_map.items():
    if value:
        os.environ[key] = str(value)

# ── 2. Reload src.config and patch the live singleton ─────────────────────────
import src.config as _cfg_module
importlib.reload(_cfg_module)

from src.config import Settings as _Settings
_cfg_module.settings = _Settings()
settings = _cfg_module.settings

# ── 3. Clear any cached LLM singletons ────────────────────────────────────────
try:
    import src.llm as _llm_module
    _llm_module.clear_llm_cache()
except Exception:
    pass

# ── 4. Configure logging ──────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.DEBUG if cfg.DEBUG_LOGS else logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
# Keep httpx / urllib3 quiet unless DEBUG_LOGS is on
if not cfg.DEBUG_LOGS:
    for _noisy in ("httpx", "httpcore", "urllib3"):
        logging.getLogger(_noisy).setLevel(logging.WARNING)

print(f"Settings applied — ENV={settings.env}, model={settings.model_name}")

## §5 — Ollama Setup (dev mode only)

Skip automatically when `cfg.ENV == "prod"`.  
On Colab / Kaggle: installs Ollama, starts the daemon, and pulls the model.  
Locally: verifies the daemon is reachable at `cfg.OLLAMA_BASE_URL`.

In [ ]:
import os
import shutil
import subprocess
import time
import requests

if cfg.ENV != "dev":
    print("prod mode — Ollama setup skipped (using OpenRouter).")
else:
    ollama_url = cfg.OLLAMA_BASE_URL

    def _ollama_alive(url: str, timeout: float = 3.0) -> bool:
        try:
            return requests.get(f"{url}/api/tags", timeout=timeout).status_code == 200
        except Exception:
            return False

    # ── Install Ollama (Colab / Kaggle only) ──────────────────────────────────
    if os.path.exists("/kaggle/working") or "google.colab" in str(globals()):
        if shutil.which("ollama") is None:
            print("Installing Ollama …")
            os.environ["PATH"] = "/usr/local/bin:" + os.environ.get("PATH", "")
            os.system("apt-get install -y -q zstd")
            os.system("curl -fsSL https://ollama.com/install.sh | sh")

            ollama_bin = shutil.which("ollama") or "/usr/local/bin/ollama"
            if not os.path.isfile(ollama_bin):
                raise RuntimeError("Ollama install failed — binary not found.")

            subprocess.Popen(
                [ollama_bin, "serve"],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
            time.sleep(3)
            print(f"Ollama installed: {shutil.which('ollama')}")
        else:
            print(f"Ollama already installed: {shutil.which('ollama')}")
    else:
        # Local environment — just verify the daemon is reachable
        if not _ollama_alive(ollama_url):
            raise RuntimeError(
                f"Ollama is not reachable at {ollama_url}.\n"
                "Start it with:  ollama serve"
            )
        print(f"Ollama already running at {ollama_url}.")

    # ── Pull model if missing ─────────────────────────────────────────────────
    model = cfg.MODEL_NAME
    tags = requests.get(f"{ollama_url}/api/tags").json().get("models", [])
    present = any(m.get("name", "").startswith(model.split(":")[0]) for m in tags)
    if not present:
        import json as _json
        print(f"Pulling {model} (first run — may take a while) …")
        resp = requests.post(
            f"{ollama_url}/api/pull", json={"name": model}, stream=True, timeout=600
        )
        for line in resp.iter_lines():
            if line:
                msg = _json.loads(line)
                status = msg.get("status", "")
                if "pulling" in status or "success" in status:
                    print(f"  {status}")
    else:
        print(f"Model '{model}' already present.")

    print("Ollama ready.")

## §6 — Dataset Download & Track A Tool Server

Downloads the HuggingFace dataset if not yet present, then starts the Track A FastAPI tool server on port 8000.

In [ ]:
from pathlib import Path

# ── Dataset download ──────────────────────────────────────────────────────────
_train_path = Path(cfg.TRAIN_FILE)
_test_path  = Path(cfg.TEST_FILE)

if _train_path.exists() and _test_path.exists():
    print(f"Dataset already present.")
    print(f"  train : {_train_path}  ({_train_path.stat().st_size // 1024} KB)")
    print(f"  test  : {_test_path}  ({_test_path.stat().st_size // 1024} KB)")
else:
    # Resolve HF_TOKEN: cfg → platform secrets → env
    _hf_token = cfg.HF_TOKEN
    if not _hf_token and PLATFORM == "colab":
        try:
            from google.colab import userdata
            _hf_token = userdata.get("HF_TOKEN") or ""
        except Exception:
            pass
    if not _hf_token and PLATFORM == "kaggle":
        try:
            from kaggle_secrets import UserSecretsClient
            _hf_token = UserSecretsClient().get_secret("HF_TOKEN") or ""
        except Exception:
            pass
    if not _hf_token:
        _hf_token = os.environ.get("HF_TOKEN", "")
    if not _hf_token:
        raise ValueError(
            "HF_TOKEN is required to download the dataset.\n"
            "Set cfg.HF_TOKEN in §3, or add a HF_TOKEN secret in Colab / Kaggle."
        )
    os.environ["HF_TOKEN"] = _hf_token
    settings.hf_token = _hf_token
    print("Downloading dataset from HuggingFace …")
    from src.download import download as _hf_download
    _hf_download()
    print("Download complete.")

# ── Track A Tool Server ───────────────────────────────────────────────────────
# Start with DATA_SPLIT=train so the server can serve both the RAG-building
# scenarios (§8) and the eval scenarios (§10), which all come from train.json.
# The server will be restarted with DATA_SPLIT=test in §14 for test inference.
_server_url    = cfg.TOOL_SERVER_URL
_server_dir    = Path("data/raw/Track A")
_server_proc   = None


def _server_healthy(url: str, timeout: float = 3.0) -> bool:
    try:
        return requests.get(url, timeout=timeout).status_code < 500
    except Exception:
        return False


def _start_server(split: str) -> subprocess.Popen:
    """Start the Track A tool server with the given DATA_SPLIT."""
    if not _server_dir.exists():
        raise FileNotFoundError(
            f"Server directory not found: {_server_dir}\n"
            "Run the dataset download block above first."
        )
    print(f"Starting Track A tool server (DATA_SPLIT={split}) …")
    env = {**os.environ, "DATA_SPLIT": split}
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "server:app",
         "--host", "0.0.0.0", "--port", "8000"],
        cwd=str(_server_dir),
        env=env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    for _ in range(20):
        if _server_healthy(_server_url):
            print(f"Tool server is up at {_server_url}  (split={split})")
            return proc
        time.sleep(1)
    proc.terminate()
    raise RuntimeError("Track A server did not become healthy within 20 s.")


if _server_healthy(_server_url):
    print(f"Tool server already running at {_server_url}")
    print("WARNING: server may be loaded with wrong DATA_SPLIT — restart if RAG errors appear.")
else:
    _server_proc = _start_server("train")

## §7 — Train / Eval Split

Splits `train.json` (which contains ground-truth labels) into:
- **RAG train slice** (80 %): fed to `TabularRAG.build_from_train()` as few-shot exemplars
- **Eval slice** (20 %): held out entirely from the RAG index and used to measure accuracy / F1

The split is stratified by `tag` so both slices contain the same proportion of `single-answer` and `multiple-answer` scenarios.

In [ ]:
import json
import random
from collections import defaultdict

# ── Load train.json ───────────────────────────────────────────────────────────
with open(cfg.TRAIN_FILE, encoding="utf-8") as fh:
    _raw = json.load(fh)

if isinstance(_raw, dict):
    _all_train = _raw.get("data") or _raw.get("scenarios") or []
else:
    _all_train = _raw

print(f"Loaded {len(_all_train)} labelled scenarios from {cfg.TRAIN_FILE}")

# ── Stratified split by tag ───────────────────────────────────────────────────
rng = random.Random(cfg.SPLIT_SEED)

_by_tag: dict[str, list] = defaultdict(list)
for s in _all_train:
    _by_tag[s.get("tag", "unknown")].append(s)

rag_train_scenarios: list = []
eval_scenarios: list      = []

for tag, group in _by_tag.items():
    _shuffled = group[:]
    rng.shuffle(_shuffled)
    n_eval = max(1, round(len(_shuffled) * cfg.EVAL_SPLIT))
    eval_scenarios.extend(_shuffled[:n_eval])
    rag_train_scenarios.extend(_shuffled[n_eval:])

print(f"\nSplit results (seed={cfg.SPLIT_SEED}, eval_split={cfg.EVAL_SPLIT:.0%}):")
print(f"  RAG train slice : {len(rag_train_scenarios)} scenarios")
print(f"  Eval slice      : {len(eval_scenarios)} scenarios")
print()

_tag_counts = {tag: {"rag": 0, "eval": 0} for tag in _by_tag}
for s in rag_train_scenarios:
    _tag_counts[s.get("tag", "unknown")]["rag"] += 1
for s in eval_scenarios:
    _tag_counts[s.get("tag", "unknown")]["eval"] += 1

print("  Tag breakdown:")
for tag, counts in sorted(_tag_counts.items()):
    print(f"    {tag:<20} rag={counts['rag']:>4}  eval={counts['eval']:>3}")

# Apply eval limit if set
if cfg.EVAL_LIMIT is not None:
    eval_scenarios = eval_scenarios[: cfg.EVAL_LIMIT]
    print(f"\nEval limit applied — using {len(eval_scenarios)} scenarios.")

## §8 — Build RAG Index

Fits the KNN index on the 80 % RAG train slice.  
Saves to `cfg.RAG_INDEX` so subsequent runs can skip this step (unless `cfg.FORCE_REBUILD_RAG = True`).

In [ ]:
import tempfile
from src.rag import TabularRAG
from src.tools.tools_track_a import TrackAClient

client_a = TrackAClient(base_url=cfg.TOOL_SERVER_URL)
rag = TabularRAG(k=cfg.RAG_K)
_rag_path = Path(cfg.RAG_INDEX)

if not cfg.FORCE_REBUILD_RAG and _rag_path.exists():
    print(f"Loading existing RAG index from {_rag_path} …")
    rag.load(str(_rag_path))
    print(f"RAG index loaded — {len(rag._entries)} entries, k={cfg.RAG_K}")
else:
    print(f"Building RAG index from {len(rag_train_scenarios)} train scenarios …")
    print("(This makes one tool-server call per scenario — may take a few minutes.)\n")

    # Persist only the rag_train_scenarios to a temp file and call build_from_train
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".json", delete=False, encoding="utf-8"
    ) as _tmp:
        json.dump(rag_train_scenarios, _tmp)
        _tmp_path = _tmp.name

    import time as _time
    _t0 = _time.perf_counter()
    rag.build_from_train(
        _tmp_path,
        client_a,
        max_examples=cfg.RAG_MAX_EXAMPLES or len(rag_train_scenarios),
    )
    _elapsed = _time.perf_counter() - _t0

    os.unlink(_tmp_path)  # clean up temp file

    rag.save(str(_rag_path))
    print(f"\nRAG index built in {_elapsed:.1f}s — {len(rag._entries)} entries indexed.")
    print(f"Saved to {_rag_path.resolve()}")

# Quick sanity check — retrieve 1 example
if rag._entries:
    _sample_fv = rag._entries[0].feature_vector
    _retrieved = rag.retrieve(_sample_fv)
    print(f"\nSanity check — retrieved {len(_retrieved)} neighbour(s) for a sample vector:")
    for _r in _retrieved:
        print(f"  [{_r.problem_type}]  answer={_r.answer}  sid={_r.scenario_id[:8]}…")

## §9 — Compile LangGraph Pipeline

In [ ]:
from src.agents.agents_track_a import build_graph_a
from src.state import make_initial_state_a

print("Compiling Track A LangGraph pipeline …")
graph_a = build_graph_a(client=client_a, rag=rag)
print("Pipeline compiled.")
print()
print("Node sequence:  retrieval → feature_extraction → rag_retrieval → analysis → validation")
print("                           ↑___________________________retry (max 2)___________________|")

## §10 — Evaluation Run — With Full Thinking Traces

Runs the pipeline on the **eval slice** (with ground-truth labels).  
Each scenario prints:
- Extracted features and classified problem type
- Few-shot RAG examples retrieved
- **Full LLM reasoning chain** (thinking trace)
- Predicted answer vs. ground-truth answer and pass/fail verdict

In [ ]:
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field as dc_field
from typing import Any


@dataclass
class EvalRecord:
    """One evaluated scenario."""
    scenario_id:  str
    tag:          str
    ground_truth: str                  # normalised answer from train.json
    prediction:   str                  # pipeline answer
    problem_type: str = ""             # classified problem type
    features:     dict = dc_field(default_factory=dict)
    rag_examples: list = dc_field(default_factory=list)
    reasoning:    str  = ""            # full LLM chain-of-thought
    elapsed_s:    float = 0.0
    exact_match:  bool  = False        # ground_truth == prediction


def _normalise_answer(raw: str) -> str:
    """Sort and deduplicate Cn codes; return pipe-separated canonical form."""
    codes = re.findall(r"C\d+", str(raw))
    if not codes:
        return ""
    return "|".join(f"C{n}" for n in sorted(set(int(c[1:]) for c in codes)))


def _run_one_eval(scenario: dict) -> EvalRecord:
    """Run the LangGraph pipeline for a single labelled scenario."""
    sid  = scenario.get("scenario_id", "")
    tag  = scenario.get("tag", "single-answer")
    gt   = _normalise_answer(scenario.get("answer", ""))

    t0 = time.perf_counter()
    initial_state = make_initial_state_a(scenario)
    final_state   = graph_a.invoke(initial_state)
    elapsed = time.perf_counter() - t0

    pred    = _normalise_answer(final_state.get("answer") or final_state.get("raw_answer", ""))
    problem = str(final_state.get("problem_type", ""))
    feats   = final_state.get("features", {})
    rag_ex  = final_state.get("rag_examples", [])
    reason  = final_state.get("reasoning", "")

    return EvalRecord(
        scenario_id  = sid,
        tag          = tag,
        ground_truth = gt,
        prediction   = pred,
        problem_type = problem,
        features     = feats,
        rag_examples = rag_ex,
        reasoning    = reason,
        elapsed_s    = elapsed,
        exact_match  = (gt == pred),
    )


def _print_trace(rec: EvalRecord, idx: int, total: int) -> None:
    """Print a formatted trace for one evaluated scenario."""
    verdict = "✓  CORRECT" if rec.exact_match else "✗  WRONG  "
    sep = "─" * 72

    print(f"\n{sep}")
    print(f"  [{idx}/{total}]  {rec.scenario_id}  |  tag={rec.tag}  |  {verdict}")
    print(sep)

    # Features
    print(f"  Problem type : {rec.problem_type}")
    _f = rec.features
    if _f:
        print(
            f"  Features     : RSRP={_f.get('serving_rsrp', '?'):.1f} dBm  "
            f"SINR={_f.get('serving_sinr', '?'):.1f} dB  "
            f"delta={_f.get('delta_db', '?'):.2f} dB  "
            f"HO_fail={_f.get('handover_failure', '?')}  "
            f"neigh_missing={_f.get('neighbor_missing', '?')}  "
            f"drop_pct={_f.get('drop_pct', 0):.0%}"
        )

    # RAG few-shot examples
    if rec.rag_examples:
        print(f"\n  ── Few-shot examples ({len(rec.rag_examples)}) ──")
        for ex in rec.rag_examples:
            for line in ex.strip().splitlines():
                print(f"    {line}")
    else:
        print("  ── No few-shot examples (RAG returned nothing) ──")

    # LLM reasoning / thinking trace
    if rec.reasoning:
        print(f"\n  ── LLM Reasoning ──")
        for line in rec.reasoning.strip().splitlines():
            print(f"    {line}")

    # Verdict
    print(f"\n  Ground truth : {rec.ground_truth}")
    print(f"  Prediction   : {rec.prediction}")
    print(f"  Time         : {rec.elapsed_s:.1f}s")


# ── Run evaluation (parallel, max 4 workers) ──────────────────────────────────
print(f"Running evaluation on {len(eval_scenarios)} scenarios …\n")
eval_records: list[EvalRecord] = []

t_total_start = time.perf_counter()

with ThreadPoolExecutor(max_workers=4) as _pool:
    _futures = {
        _pool.submit(_run_one_eval, s): i
        for i, s in enumerate(eval_scenarios, 1)
    }
    _completed: dict[int, EvalRecord] = {}
    for _fut in as_completed(_futures):
        _idx = _futures[_fut]
        try:
            _completed[_idx] = _fut.result()
        except Exception as _exc:
            s = eval_scenarios[_idx - 1]
            print(f"[ERROR] scenario {s.get('scenario_id', '')} failed: {_exc}")
            _completed[_idx] = EvalRecord(
                scenario_id  = s.get("scenario_id", ""),
                tag          = s.get("tag", ""),
                ground_truth = _normalise_answer(s.get("answer", "")),
                prediction   = "",
                exact_match  = False,
            )

# Restore order and print traces
for _idx in sorted(_completed):
    rec = _completed[_idx]
    eval_records.append(rec)
    if cfg.SHOW_THINKING:
        _print_trace(rec, _idx, len(eval_scenarios))

t_total = time.perf_counter() - t_total_start
print(f"\n{'─'*72}")
print(f"Evaluation complete — {len(eval_records)} scenarios in {t_total:.1f}s "
      f"({t_total / max(len(eval_records), 1):.1f}s/scenario)")

## §11 — Evaluation Metrics

Computes three complementary metrics:

| Metric | Definition |
|--------|------------|
| **Exact-Match Accuracy** | Fraction of scenarios where predicted code set == ground-truth code set |
| **Macro F1** | Average per-scenario F1 (treats each scenario's code set as a binary classification task) |
| **Micro F1** | Aggregated TP/FP/FN across all codes (good for imbalanced multi-label distributions) |

All metrics are reported **overall** and broken down by **tag** (`single-answer` / `multiple-answer`).

In [ ]:
import pandas as pd


# ── Helper: parse a pipe-separated answer string into a frozenset of codes ────
def _code_set(answer: str) -> frozenset[str]:
    return frozenset(re.findall(r"C\d+", str(answer)))


# ── Per-scenario F1 ───────────────────────────────────────────────────────────
def _scenario_f1(gt_set: frozenset, pred_set: frozenset) -> float:
    """Binary F1 for one scenario (micro over its codes)."""
    tp = len(gt_set & pred_set)
    fp = len(pred_set - gt_set)
    fn = len(gt_set - pred_set)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


# ── Build results DataFrame ───────────────────────────────────────────────────
df_eval = pd.DataFrame([
    {
        "scenario_id":   r.scenario_id,
        "tag":           r.tag,
        "problem_type":  r.problem_type,
        "ground_truth":  r.ground_truth,
        "prediction":    r.prediction,
        "exact_match":   r.exact_match,
        "elapsed_s":     r.elapsed_s,
        "f1":            _scenario_f1(_code_set(r.ground_truth), _code_set(r.prediction)),
    }
    for r in eval_records
])

# ── Aggregate metrics ─────────────────────────────────────────────────────────
def _micro_f1(df: pd.DataFrame) -> float:
    """Micro F1 aggregated across all scenarios in df."""
    tp = fp = fn = 0
    for _, row in df.iterrows():
        gt_set   = _code_set(row["ground_truth"])
        pred_set = _code_set(row["prediction"])
        tp += len(gt_set & pred_set)
        fp += len(pred_set - gt_set)
        fn += len(gt_set - pred_set)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def _print_metric_block(label: str, df: pd.DataFrame) -> None:
    n         = len(df)
    acc       = df["exact_match"].mean()
    macro_f1  = df["f1"].mean()
    micro_f1v = _micro_f1(df)
    coverage  = df["prediction"].str.strip().ne("").mean()
    print(f"  {label:<26}  n={n:>4}  "
          f"Accuracy={acc:.3f}  MacroF1={macro_f1:.3f}  MicroF1={micro_f1v:.3f}  "
          f"Coverage={coverage:.3f}")


print("=" * 80)
print("  §11 — Evaluation Metrics")
print("=" * 80)
print()

_print_metric_block("Overall", df_eval)
print()

for _tag in sorted(df_eval["tag"].unique()):
    _print_metric_block(f"tag = {_tag}", df_eval[df_eval["tag"] == _tag])

print()
print(f"  Total scenarios  : {len(df_eval)}")
print(f"  Exact matches    : {df_eval['exact_match'].sum()}")
print(f"  Mean latency     : {df_eval['elapsed_s'].mean():.1f}s / scenario")
print("=" * 80)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Accuracy bar chart by tag ─────────────────────────────────────────────────
_tag_metrics = (
    df_eval.groupby("tag")
    .agg(
        Accuracy   = ("exact_match", "mean"),
        Macro_F1   = ("f1",          "mean"),
        Count      = ("scenario_id", "count"),
    )
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Track A — Evaluation Metrics by Tag", fontsize=13, fontweight="bold")

# Left: accuracy
ax0 = axes[0]
bars = ax0.bar(_tag_metrics["tag"], _tag_metrics["Accuracy"], color=["#4C72B0", "#DD8452"], width=0.5)
ax0.set_ylim(0, 1.05)
ax0.set_ylabel("Exact-Match Accuracy")
ax0.set_title("Accuracy")
ax0.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
for bar, val in zip(bars, _tag_metrics["Accuracy"]):
    ax0.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
             f"{val:.1%}", ha="center", va="bottom", fontsize=10)

# Right: Macro F1
ax1 = axes[1]
bars2 = ax1.bar(_tag_metrics["tag"], _tag_metrics["Macro_F1"], color=["#4C72B0", "#DD8452"], width=0.5)
ax1.set_ylim(0, 1.05)
ax1.set_ylabel("Macro F1")
ax1.set_title("Macro F1")
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
for bar, val in zip(bars2, _tag_metrics["Macro_F1"]):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
             f"{val:.1%}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()
print(_tag_metrics.to_string(index=False))

## §12 — Per-Problem-Type Analysis

Breaks down accuracy and F1 for each problem type classified by the feature extractor (`LATE_HANDOVER`, `NEIGHBOR_MISSING`, `UNKNOWN`, etc.).

In [ ]:
_ptype_metrics = (
    df_eval.groupby("problem_type")
    .agg(
        Count      = ("scenario_id", "count"),
        Accuracy   = ("exact_match", "mean"),
        Macro_F1   = ("f1",          "mean"),
        Avg_time_s = ("elapsed_s",   "mean"),
    )
    .sort_values("Count", ascending=False)
    .reset_index()
)

print("=" * 75)
print("  §12 — Metrics by Problem Type")
print("=" * 75)
print(
    _ptype_metrics.to_string(
        index=False,
        formatters={
            "Accuracy":   "{:.3f}".format,
            "Macro_F1":   "{:.3f}".format,
            "Avg_time_s": "{:.1f}".format,
        },
    )
)
print("=" * 75)

# ── Horizontal bar chart ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, max(3, len(_ptype_metrics) * 0.7 + 1)))

_types  = _ptype_metrics["problem_type"].tolist()
_acc    = _ptype_metrics["Accuracy"].tolist()
_f1     = _ptype_metrics["Macro_F1"].tolist()
_y      = range(len(_types))

ax.barh([y + 0.2 for y in _y], _acc, height=0.35, label="Accuracy",  color="#4C72B0")
ax.barh([y - 0.2 for y in _y], _f1,  height=0.35, label="Macro F1",  color="#55A868")

ax.set_yticks(list(_y))
ax.set_yticklabels(_types)
ax.set_xlim(0, 1.1)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_xlabel("Score")
ax.set_title("Track A — Accuracy & Macro F1 by Problem Type")
ax.legend()

for y, acc, f1v in zip(_y, _acc, _f1):
    ax.text(acc + 0.01, y + 0.2, f"{acc:.1%}", va="center", fontsize=8)
    ax.text(f1v + 0.01, y - 0.2, f"{f1v:.1%}", va="center", fontsize=8)

plt.tight_layout()
plt.show()

## §13 — Error Analysis

Inspect the **wrong predictions** in detail: what the model predicted, what the ground truth was, and which feature pattern led to the mistake.

In [ ]:
df_errors = df_eval[~df_eval["exact_match"]].copy()

print(f"Wrong predictions: {len(df_errors)} / {len(df_eval)}  "
      f"({len(df_errors)/max(len(df_eval),1):.1%} error rate)\n")

if df_errors.empty:
    print("No errors — perfect accuracy on the eval set!")
else:
    # ── Confusion heat-map: ground_truth → prediction ─────────────────────────
    _gt_counts   = df_errors.groupby(["ground_truth", "prediction"]).size().reset_index(name="count")
    print("Ground-truth → Prediction confusion (errors only):")
    print(_gt_counts.to_string(index=False))
    print()

    # ── Detailed error listing ─────────────────────────────────────────────────
    _err_records = [r for r in eval_records if not r.exact_match]
    for i, rec in enumerate(_err_records, 1):
        sep = "─" * 72
        print(f"\n{sep}")
        print(f"  Error {i}/{len(_err_records)}  |  {rec.scenario_id}  |  tag={rec.tag}")
        print(f"  Problem type : {rec.problem_type}")
        _f = rec.features
        if _f:
            print(
                f"  Features     : RSRP={_f.get('serving_rsrp', '?'):.1f}  "
                f"SINR={_f.get('serving_sinr', '?'):.1f}  "
                f"delta={_f.get('delta_db', '?'):.2f}  "
                f"HO_fail={_f.get('handover_failure', '?')}  "
                f"neigh_missing={_f.get('neighbor_missing', '?')}  "
                f"drop_pct={_f.get('drop_pct', 0):.0%}"
            )
        print(f"  Ground truth : {rec.ground_truth}")
        print(f"  Prediction   : {rec.prediction}")
        if rec.rag_examples:
            print(f"\n  ── Few-shot examples ──")
            for ex in rec.rag_examples:
                for line in ex.strip().splitlines():
                    print(f"    {line}")
        print(f"\n  ── LLM Reasoning ──")
        for line in rec.reasoning.strip().splitlines():
            print(f"    {line}")

## §14 — Full Test-Set Inference

Runs the pipeline on `test.json` (no ground-truth labels).  
Uses the full RAG index built from all of `rag_train_scenarios`.  
Results are stored in `test_results` and exported in §15.

In [ ]:
# ── Load test.json ────────────────────────────────────────────────────────────
with open(cfg.TEST_FILE, encoding="utf-8") as fh:
    _raw_test = json.load(fh)

if isinstance(_raw_test, dict):
    test_scenarios = (
        _raw_test.get("data")
        or _raw_test.get("scenarios")
        or _raw_test.get("questions")
        or []
    )
else:
    test_scenarios = _raw_test

# Tag each scenario as Track A (needed by make_initial_state_a)
for s in test_scenarios:
    s.setdefault("track", "A")

if cfg.TEST_LIMIT is not None:
    test_scenarios = test_scenarios[: cfg.TEST_LIMIT]

print(f"Test set: {len(test_scenarios)} scenarios loaded from {cfg.TEST_FILE}")

# ── Run inference ─────────────────────────────────────────────────────────────
test_results: list[dict] = []

print(f"Running inference …")
_t0 = time.perf_counter()

with ThreadPoolExecutor(max_workers=4) as _pool:
    def _infer(scenario: dict) -> dict:
        sid           = scenario.get("scenario_id", "")
        initial_state = make_initial_state_a(scenario)
        final_state   = graph_a.invoke(initial_state)
        answer        = _normalise_answer(
            final_state.get("answer") or final_state.get("raw_answer", "")
        )
        return {"ID": sid, "Track A": answer, "Track B": ""}

    _fut_to_s = {_pool.submit(_infer, s): s for s in test_scenarios}
    for _fut in as_completed(_fut_to_s):
        s = _fut_to_s[_fut]
        try:
            row = _fut.result()
        except Exception as exc:
            row = {"ID": s.get("scenario_id", ""), "Track A": "", "Track B": ""}
            print(f"[ERROR] {s.get('scenario_id', '')}: {exc}")
        test_results.append(row)
        print(f"  {row['ID']} → {row['Track A']}")

_elapsed_test = time.perf_counter() - _t0
print(f"\nInference complete — {len(test_results)} scenarios in {_elapsed_test:.1f}s")

# ── Quick coverage summary ────────────────────────────────────────────────────
df_test = pd.DataFrame(test_results)
_answered = df_test["Track A"].str.strip().ne("").sum()
print(f"Coverage: {_answered}/{len(df_test)}  ({_answered/max(len(df_test),1):.1%})")

## §15 — Export Results

Writes `result_track_a.csv` in the submission format expected by the competition (`ID`, `Track A`, `Track B`).  
On Colab / Kaggle a browser download / output copy is triggered automatically.

In [ ]:
import csv

output_path = Path(cfg.OUTPUT_CSV)

# Ensure canonical column order and sort by ID
df_out = df_test[["ID", "Track A", "Track B"]].sort_values("ID")

with output_path.open("w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=["ID", "Track A", "Track B"])
    writer.writeheader()
    writer.writerows(df_out.to_dict("records"))

print(f"Results written → {output_path.resolve()}")
print(f"  Rows       : {len(df_out)}")
print(f"  Coverage   : {df_out['Track A'].str.strip().ne('').sum()} non-empty answers")

# ── Platform download helpers ─────────────────────────────────────────────────
if PLATFORM == "colab":
    from google.colab import files
    files.download(str(output_path))
    print("Browser download triggered.")
elif PLATFORM == "kaggle":
    import shutil as _shutil
    _dst = Path("/kaggle/working") / output_path.name
    _shutil.copy(output_path, _dst)
    print(f"Copied to Kaggle output: {_dst}")
else:
    print("Local run — no browser download needed.")

# ── Preview first 10 rows ─────────────────────────────────────────────────────
print()
print(df_out.head(10).to_string(index=False))

## §16 — (Optional) Cleanup

Stop the Track A tool server background process started in §6.

In [ ]:
if "_server_proc" in dir() and _server_proc is not None:
    _server_proc.terminate()
    try:
        _server_proc.wait(timeout=5)
    except Exception:
        _server_proc.kill()
    print("Track A tool server stopped.")
else:
    print("No server process to stop (was already running before this session).")